# Revisão do M2

Este notebook verifica o M2 atual e orienta sua refatoração para responder melhor à RQ1 do paper_v8.

**Conclusão principal:** o cálculo atual é simples, auditável e não usa LLM. O M2 continua útil porque mede percepções específicas sobre seis papéis de engenharia de software, informação que o M1 reformulado não substitui. Porém, o enunciado “extinta ou severamente afetada” combina dois fenômenos diferentes e o resultado agregado por corte ignora vínculos disponíveis entre os mesmos respondentes.

## De onde vêm os dados do M2?

### Resposta curta

O M2 vem de **seis perguntas estruturadas do survey**, uma para cada papel: Backend, Frontend, QA, Project Manager, Product Manager e Scrum Master.

Cada estudante informa concordância de 1 a 5 com a afirmação de que o papel será “extinto ou severamente afetado” pela IA generativa:

- 1: discordo totalmente;
- 5: concordo totalmente.

O M2 resume as respostas por `Semestre × temporal_marker × papel`. Não usa Git, transcrições ou chamadas LLM.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
LAKE_PATH = PROJECT_ROOT / "data/lake/student_responses.parquet"
M2_PATH = PROJECT_ROOT / "paper_v8/data/m2_role_disruption_risk.csv"
METRICS_DIR = PROJECT_ROOT / "paper_v8/scripts/metrics"

if str(METRICS_DIR) not in sys.path:
    sys.path.insert(0, str(METRICS_DIR))

from m2_role_disruption_risk import ROLE_QUESTION_COLUMNS

student_responses_df = pd.read_parquet(LAKE_PATH)
m2_original_df = pd.read_csv(M2_PATH, dtype={"Semestre": str})

provenance_df = pd.DataFrame(
    [
        {
            "etapa": "1. Respostas estruturadas",
            "arquivo": "data/lake/student_responses.parquet",
            "linhas": len(student_responses_df),
            "unidade": "submissão de estudante",
        },
        {
            "etapa": "2. M2 publicado",
            "arquivo": "paper_v8/data/m2_role_disruption_risk.csv",
            "linhas": len(m2_original_df),
            "unidade": "semestre × corte × papel",
        },
    ]
)
provenance_df

,etapa,arquivo,linhas,unidade
0,1. Respostas estruturadas,data/lake/student_responses.parquet,187,submissão de estudante
1,2. M2 publicado,paper_v8/data/m2_role_disruption_risk.csv,36,semestre × corte × papel


### Como o valor é produzido

Para cada papel $r$, semestre $s$ e corte $c$, o M2 preserva a distribuição das respostas Likert:

$$
M2_{r,s,c}=\operatorname{Resumo}\left(\{x_{i,r,s,c}\}\right)
$$

O resumo contém $n$, média, desvio-padrão, mediana, quartis, IQR e moda.

**Unidade do M2 publicado:** coorte por corte e papel (`Semestre × temporal_marker × role`). O CSV não preserva o vínculo individual, embora o arquivo de origem possua um identificador que permite análise pareada sob controle de privacidade.

## Os valores do M2 estão corretamente calculados?

### Resposta curta

A auditoria abaixo verifica:

- presença das seis perguntas;
- ausência de valores inválidos ou ausentes;
- uma resposta por estudante, corte e papel;
- recálculo de $n$, média, desvio-padrão, mediana, quartis e IQR;
- igualdade entre o recálculo e o CSV publicado.

Isso avalia a implementação. A validade do enunciado e sua adequação à RQ1 são discutidas separadamente.

In [2]:
from IPython.display import display

KEYS = ["Semestre", "temporal_marker", "role"]

raw_role_df = pd.concat(
    [
        student_responses_df[
            ["Semestre", "temporal_marker", "Email Address", column]
        ].rename(columns={column: "score"}).assign(role=role)
        for role, column in ROLE_QUESTION_COLUMNS.items()
    ],
    ignore_index=True,
)
raw_role_df["score"] = pd.to_numeric(raw_role_df["score"], errors="coerce")

recalculated_df = (
    raw_role_df.groupby(KEYS)["score"]
    .agg(
        n_total="size",
        n_valid="count",
        mean="mean",
        std="std",
        median="median",
        q1=lambda values: values.quantile(0.25),
        q3=lambda values: values.quantile(0.75),
    )
    .reset_index()
)
recalculated_df["n_missing"] = (
    recalculated_df["n_total"] - recalculated_df["n_valid"]
)
recalculated_df["iqr"] = recalculated_df["q3"] - recalculated_df["q1"]

COMPARE_COLUMNS = [
    "n_total", "n_valid", "n_missing", "mean", "std", "median", "q1", "q3", "iqr"
]
comparison_df = recalculated_df.merge(
    m2_original_df[KEYS + COMPARE_COLUMNS],
    on=KEYS,
    suffixes=("_recalculado", "_publicado"),
    validate="one_to_one",
)
max_numeric_error = max(
    (
        comparison_df[f"{column}_recalculado"]
        - comparison_df[f"{column}_publicado"]
    ).abs().max()
    for column in COMPARE_COLUMNS
)

respondent_key = raw_role_df["Email Address"].astype("string").str.strip().str.lower()
audit_df = pd.DataFrame(
    [
        {"verificação": "As seis perguntas existem", "resultado": len(ROLE_QUESTION_COLUMNS) == 6},
        {"verificação": "Sem respostas ausentes", "resultado": raw_role_df["score"].notna().all()},
        {"verificação": "Todos os valores são inteiros entre 1 e 5", "resultado": raw_role_df["score"].between(1, 5).all() and raw_role_df["score"].mod(1).eq(0).all()},
        {"verificação": "Sem estudante-corte-papel duplicado", "resultado": not raw_role_df.assign(_respondent=respondent_key).duplicated(["Semestre", "temporal_marker", "_respondent", "role"]).any()},
        {"verificação": "M2 publicado confere com o recálculo", "resultado": max_numeric_error < 1e-12},
    ]
)

display(audit_df)
pd.Series({"erro numérico máximo": max_numeric_error, "linhas comparadas": len(comparison_df)})

,verificação,resultado
0,As seis perguntas existem,True
1,Sem respostas ausentes,True
2,Todos os valores são inteiros entre 1 e 5,True
3,Sem estudante-corte-papel duplicado,True
4,M2 publicado confere com o recálculo,True


erro numérico máximo    4.440892e-16
linhas comparadas       3.600000e+01
dtype: float64

## O M2 responde adequadamente à RQ1?

> **RQ1:** How do student developers utilize Generative AI, and how does this adoption shape their longitudinal perceptions of software engineering roles and project expectations?

### Resposta curta

O M2 responde **parcialmente** à parte sobre percepções dos papéis profissionais. Ele mostra quanto os estudantes concordam que seis papéis serão extintos ou severamente afetados pela IA.

O M2 não responde:

- como os estudantes usam IA;
- se a adoção de IA explica as percepções;
- qual mecanismo de mudança é esperado em cada papel;
- se a resposta significa extinção, transformação de tarefas ou apenas aumento de produtividade.

Portanto, o M2 deve ser descrito como **concordância com disrupção percebida por papel**, não como risco objetivo de extinção.

### O novo M1 torna o M2 desnecessário?

**Não.** O M1c proposto no notebook do M1 captura temas gerais de impacto na carreira: produtividade, competências, substituição, risco e incerteza. O M2 acrescenta comparação estruturada entre papéis específicos.

As métricas são complementares:

- **M1c:** explica *como* os estudantes imaginam a transformação profissional;
- **M2:** compara *quais papéis* são percebidos como mais ou menos afetados.

Não é recomendado fundi-las em uma média. A relação entre elas deve aparecer na interpretação: temas qualitativos de M1c ajudam a explicar os padrões estruturados de M2.

In [3]:
category_distribution_df = (
    raw_role_df.groupby(KEYS + ["score"])
    .size()
    .rename("count")
    .reset_index()
)
category_distribution_df["share"] = category_distribution_df.groupby(KEYS)[
    "count"
].transform(lambda values: values / values.sum())

role_summary_df = (
    raw_role_df.assign(
        discordancia=raw_role_df["score"].isin([1, 2]),
        neutro=raw_role_df["score"].eq(3),
        concordancia=raw_role_df["score"].isin([4, 5]),
    )
    .groupby(KEYS, as_index=False)
    .agg(
        n=("score", "size"),
        media=("score", "mean"),
        mediana=("score", "median"),
        iqr=("score", lambda values: values.quantile(0.75) - values.quantile(0.25)),
        proporcao_1_2=("discordancia", "mean"),
        proporcao_3=("neutro", "mean"),
        proporcao_4_5=("concordancia", "mean"),
    )
)
role_summary_df.round(3)

,Semestre,temporal_marker,role,n,media,mediana,iqr,proporcao_1_2,proporcao_3,proporcao_4_5
0,2025.2,T1,Backend,46,2.391,2.0,1.00,0.565,0.326,0.109
1,2025.2,T1,Frontend,46,2.739,3.0,1.00,0.435,0.326,0.239
2,2025.2,T1,Product Manager,46,2.087,2.0,2.00,0.630,0.304,0.065
3,2025.2,T1,Project Manager,46,1.826,1.5,1.00,0.761,0.174,0.065
4,2025.2,T1,QA,46,2.065,2.0,1.00,0.761,0.130,0.109
5,2025.2,T1,Scrum Master,46,2.326,2.0,1.00,0.587,0.304,0.109
6,2025.2,T2,Backend,41,2.268,2.0,1.00,0.610,0.317,0.073
7,2025.2,T2,Frontend,41,2.683,3.0,2.00,0.463,0.268,0.268
8,2025.2,T2,Product Manager,41,1.976,2.0,2.00,0.707,0.220,0.073
9,2025.2,T2,Project Manager,41,1.829,2.0,1.00,0.756,0.220,0.024


In [4]:
import plotly.express as px
from IPython.display import HTML

role_trajectory_figure = px.line(
    role_summary_df,
    x="temporal_marker",
    y="media",
    color="role",
    facet_col="Semestre",
    markers=True,
    category_orders={"temporal_marker": ["T1", "T2", "T3"]},
    labels={
        "temporal_marker": "Corte",
        "media": "Concordância média (1–5)",
        "role": "Papel",
    },
    title="M2 legado por papel, semestre e corte",
)
role_trajectory_figure.update_yaxes(range=[1, 5])
display(HTML(role_trajectory_figure.to_html(include_plotlyjs="cdn", full_html=False)))

## A análise pode ser longitudinal?

O arquivo de origem contém um identificador estável. Ele não deve aparecer nos resultados, mas permite parear respostas do mesmo estudante.

A análise publicada compara médias de cortes com tamanhos diferentes. Isso é uma comparação repetida de coortes, não necessariamente mudança individual.

A análise pareada abaixo inclui somente estudantes observados em T1 e T3 e reporta, por papel:

- número de pares;
- médias em T1 e T3;
- média e mediana da mudança individual;
- proporções que aumentaram, não mudaram ou diminuíram.

In [5]:
panel_source_df = raw_role_df.assign(
    respondent=raw_role_df["Email Address"].astype("string").str.strip().str.lower()
)

respondent_coverage_df = (
    panel_source_df[["Semestre", "temporal_marker", "respondent"]]
    .drop_duplicates()
    .groupby(["Semestre", "respondent"])["temporal_marker"]
    .nunique()
    .groupby("Semestre")
    .agg(
        respondentes="size",
        completos_T1_T2_T3=lambda values: values.eq(3).sum(),
    )
    .reset_index()
)

t1_t3_panel_df = (
    panel_source_df.pivot(
        index=["Semestre", "respondent", "role"],
        columns="temporal_marker",
        values="score",
    )
    .dropna(subset=["T1", "T3"])
    .reset_index()
)
t1_t3_panel_df["mudanca_T1_T3"] = t1_t3_panel_df["T3"] - t1_t3_panel_df["T1"]

paired_change_df = (
    t1_t3_panel_df.groupby(["Semestre", "role"], as_index=False)
    .agg(
        pares=("mudanca_T1_T3", "size"),
        media_T1=("T1", "mean"),
        media_T3=("T3", "mean"),
        mudanca_media=("mudanca_T1_T3", "mean"),
        mudanca_mediana=("mudanca_T1_T3", "median"),
        proporcao_aumentou=("mudanca_T1_T3", lambda values: values.gt(0).mean()),
        proporcao_igual=("mudanca_T1_T3", lambda values: values.eq(0).mean()),
        proporcao_diminuiu=("mudanca_T1_T3", lambda values: values.lt(0).mean()),
    )
)

display(respondent_coverage_df)
paired_change_df.round(3)

,Semestre,respondentes,completos_T1_T2_T3
0,2025.2,50,34
1,2026.1,24,16


,Semestre,role,pares,media_T1,media_T3,mudanca_media,mudanca_mediana,proporcao_aumentou,proporcao_igual,proporcao_diminuiu
0,2025.2,Backend,38,2.342,2.316,-0.026,0.0,0.237,0.447,0.316
1,2025.2,Frontend,38,2.737,2.763,0.026,0.0,0.316,0.316,0.368
2,2025.2,Product Manager,38,2.026,2.132,0.105,0.0,0.342,0.395,0.263
3,2025.2,Project Manager,38,1.763,2.079,0.316,0.0,0.368,0.526,0.105
4,2025.2,QA,38,2.000,2.053,0.053,0.0,0.263,0.500,0.237
5,2025.2,Scrum Master,38,2.289,2.421,0.132,0.0,0.289,0.500,0.211
6,2026.1,Backend,16,2.375,2.688,0.312,0.0,0.250,0.500,0.250
7,2026.1,Frontend,16,3.062,3.125,0.062,0.0,0.250,0.438,0.312
8,2026.1,Product Manager,16,2.438,2.500,0.062,0.0,0.188,0.375,0.438
9,2026.1,Project Manager,16,2.375,2.312,-0.062,-0.5,0.250,0.250,0.500


## As diferenças entre papéis são robustas?

Rankings por média são fáceis de comunicar, mas não devem substituir as distribuições. Diferenças pequenas podem coexistir com forte sobreposição entre respostas.

A tabela abaixo mostra a ordenação dentro de cada semestre e corte. Ela deve ser interpretada junto com mediana, IQR e proporções 1–2, 3 e 4–5.

In [6]:
role_ranking_df = role_summary_df[
    ["Semestre", "temporal_marker", "role", "media", "mediana", "iqr", "proporcao_4_5"]
].copy()
role_ranking_df["rank_media"] = role_ranking_df.groupby(
    ["Semestre", "temporal_marker"]
)["media"].rank(method="min", ascending=False)
role_ranking_df.sort_values(
    ["Semestre", "temporal_marker", "rank_media", "role"]
).round(3)

,Semestre,temporal_marker,role,media,mediana,iqr,proporcao_4_5,rank_media
1,2025.2,T1,Frontend,2.739,3.0,1.00,0.239,1.0
0,2025.2,T1,Backend,2.391,2.0,1.00,0.109,2.0
5,2025.2,T1,Scrum Master,2.326,2.0,1.00,0.109,3.0
2,2025.2,T1,Product Manager,2.087,2.0,2.00,0.065,4.0
4,2025.2,T1,QA,2.065,2.0,1.00,0.109,5.0
3,2025.2,T1,Project Manager,1.826,1.5,1.00,0.065,6.0
7,2025.2,T2,Frontend,2.683,3.0,2.00,0.268,1.0
6,2025.2,T2,Backend,2.268,2.0,1.00,0.073,2.0
11,2025.2,T2,Scrum Master,2.122,2.0,2.00,0.049,3.0
10,2025.2,T2,QA,2.098,2.0,2.00,0.049,4.0


In [7]:
cross_sectional_change_df = (
    role_summary_df.pivot(
        index=["Semestre", "role"],
        columns="temporal_marker",
        values="media",
    )
    .assign(mudanca_agregada=lambda frame: frame["T3"] - frame["T1"])
    .reset_index()
)

change_comparison_df = cross_sectional_change_df.merge(
    paired_change_df[["Semestre", "role", "pares", "mudanca_media", "mudanca_mediana"]],
    on=["Semestre", "role"],
    validate="one_to_one",
).rename(columns={"mudanca_media": "mudanca_pareada_media"})
change_comparison_df.round(3)

,Semestre,role,T1,T2,T3,mudanca_agregada,pares,mudanca_pareada_media,mudanca_mediana
0,2025.2,Backend,2.391,2.268,2.300,-0.091,38,-0.026,0.0
1,2025.2,Frontend,2.739,2.683,2.725,-0.014,38,0.026,0.0
2,2025.2,Product Manager,2.087,1.976,2.150,0.063,38,0.105,0.0
3,2025.2,Project Manager,1.826,1.829,2.075,0.249,38,0.316,0.0
4,2025.2,QA,2.065,2.098,2.050,-0.015,38,0.053,0.0
5,2025.2,Scrum Master,2.326,2.122,2.425,0.099,38,0.132,0.0
6,2026.1,Backend,2.318,2.100,2.500,0.182,16,0.312,0.0
7,2026.1,Frontend,3.136,2.650,3.278,0.141,16,0.062,0.0
8,2026.1,Product Manager,2.500,2.100,2.333,-0.167,16,0.062,0.0
9,2026.1,Project Manager,2.455,2.000,2.167,-0.288,16,-0.062,-0.5


### Resultado nos dados atuais

- Há 34 respondentes completos em T1–T2–T3 em 2025.2 e 16 em 2026.1.
- A comparação T1–T3 possui 38 pares em 2025.2 e 16 em 2026.1.
- Em 2025.2, a mediana da mudança individual é zero para todos os papéis.
- Em 2026.1, as medianas são zero para Backend, Frontend e Product Manager; +0,5 para QA; −0,5 para Project Manager e Scrum Master.
- Algumas mudanças agregadas e pareadas têm sinais diferentes, demonstrando efeito de composição.

Não há uma trajetória única compartilhada pelos seis papéis. Os resultados devem permanecer separados por papel e semestre.

## O enunciado atual mede um único construto?

**Não.** “O papel será extinto ou severamente afetado” é uma pergunta dupla:

- um estudante pode considerar que o papel continuará existindo, mas será muito transformado;
- outro pode interpretar “afetado” como melhoria de produtividade;
- ambos podem marcar o mesmo valor por razões opostas.

O escore deve ser chamado de **concordância com disrupção do papel**, não probabilidade de extinção.

Em uma próxima coleta, separar:

1. probabilidade percebida de extinção do papel;
2. magnitude esperada de transformação das tarefas;
3. direção esperada do impacto: substituição, complementação ou criação de oportunidades;
4. mudança esperada nas competências necessárias.

Esses itens não devem ser combinados automaticamente em um único escore.

## A escala e as estatísticas são adequadas?

A escala Likert 1–5 é adequada para concordância e não precisa mudar para 0–10. Mais categorias adicionariam precisão aparente, não necessariamente informação.

A apresentação principal deve incluir:

- proporção de discordância (1–2);
- proporção neutra (3);
- proporção de concordância (4–5);
- mediana e IQR;
- tamanho da amostra;
- média e desvio-padrão apenas como complemento.

Para mudança temporal, reportar a distribuição de $T3-T1$ entre os mesmos respondentes: aumentou, permaneceu igual ou diminuiu. Rankings de médias são secundários porque as distribuições dos papéis se sobrepõem.

## Agrupar por semestre ou apenas por T1, T2 e T3?

A análise principal deve manter `Semestre × temporal_marker × papel`, pois os semestres são coortes diferentes e apresentam níveis distintos.

Para mudança temporal:

- **principal:** análise pareada T1–T3 dentro de cada semestre;
- **secundária:** distribuições de todos os respondentes em T1, T2 e T3;
- **opcional:** resultado agregado entre semestres, claramente identificado como resumo descritivo.

Não se deve usar apenas a média agregada entre semestres nem interpretar cortes com participantes diferentes como trajetória individual.

## Decisão de refatoração

**Manter o M2 separado do M1**, mas alterar sua interpretação e sua análise.

### Com os dados atuais

1. Renomear para **M2 legado — Concordância com disrupção percebida por papel**.
2. Preservar os seis papéis separadamente.
3. Usar distribuições completas, não apenas médias e rankings.
4. Adicionar mudança pareada T1–T3 por semestre.
5. Manter resultados agregados como análise secundária.
6. Não interpretar o escore como risco objetivo de extinção.

### Em uma próxima coleta

Substituir a pergunta dupla por um perfil de percepção dos papéis:

- **M2a — Extinção percebida**;
- **M2b — Transformação percebida das tarefas**;
- **M2c — Direção do impacto**: substituição, complementação ou novas oportunidades;
- **M2d — Mudança percebida nas competências**.

O papel preferido do respondente pode apoiar uma análise exploratória de percepção do próprio papel, mas somente quando houver tamanho mínimo por grupo.

## Decisão sobre a pipeline

### Regeneração imediata

É recomendado regenerar os artefatos analíticos do M2 a partir de `student_responses.parquet` para incluir:

- distribuições 1–5 e proporções 1–2, 3 e 4–5;
- resumos por semestre, corte e papel;
- mudanças pareadas T1–T3;
- contagens de participantes completos e perdas entre cortes.

Essa regeneração é **determinística e não requer chamadas LLM**.

O identificador de pareamento deve ser pseudonimizado antes da análise e nunca aparecer em artefatos públicos. O e-mail bruto não deve ser escrito nos novos CSVs.

### Nova coleta

Separar extinção, transformação, direção do impacto e competências exige alterar o survey e coletar novas respostas. Uma LLM não pode recuperar retrospectivamente qual parte da pergunta dupla motivou uma resposta de 1 a 5.

### Critérios de aceite

- contrato de saída versionado;
- validação da faixa 1–5 e da completude;
- nenhuma informação identificável no artefato analítico;
- distribuições e $n$ preservados;
- análise pareada separada da análise agregada;
- M2 legado preservado sem sobrescrita;
- interpretação limitada à percepção autorrelatada.

## Correções necessárias no paper_v8

Após a refatoração, revisar os seguintes pontos sem alterar o LaTeX antes de congelar os novos resultados:

1. **Metodologia:** trocar “Role-Disruption Risk” por uma formulação que deixe claro que o valor é concordância autorrelatada, não risco objetivo.
2. **Definição do M2:** explicitar que o item combina “extinta” e “severamente afetada” e registrar essa limitação de construto.
3. **Resultados:** apresentar distribuições e proporções de concordância, não apenas médias response-weighted.
4. **Temporalidade:** incluir resultados pareados por semestre e não chamar comparação de amostras diferentes de trajetória individual.
5. **Ranking:** qualificar a afirmação de que Frontend é sempre o mais alto; em 2026.1/T1 há empate de média com Scrum Master.
6. **Forma temporal:** remover a afirmação geral de que todos os papéis seguem a mesma trajetória não monotônica; isso vale para o resumo combinado, não para todas as análises por coorte e respondente.
7. **RQ1:** não afirmar que M2 demonstra que adoção de IA moldou percepções. M2 descreve percepções; a relação com adoção exige análise conjunta com indicadores de uso.
8. **Tabela de métricas:** manter M1 e M2 separados, explicando que M1c fornece mecanismos temáticos e M2 compara papéis específicos.
9. **Ameaças à validade:** adicionar ambiguidade do item, atrito da amostra entre cortes e sensibilidade à agregação.

## Resumo das mudanças propostas

| Aspecto | Situação atual | Mudança proposta | Ação necessária |
|---|---|---|---|
| Relação com a RQ1 | Descreve “risco” dos papéis | Descrever concordância com disrupção percebida | Corrigir interpretação |
| Relação com M1 | Métricas apresentadas lado a lado | Manter separadas e complementares | Integrar apenas na discussão |
| Enunciado | “Extinta ou severamente afetada” | Separar extinção, transformação, direção e competências | Nova coleta |
| Estatística | Média, mediana e ranking | Distribuições, proporções, mediana, IQR e $n$ | Regenerar deterministicamente |
| Temporalidade | Médias de cortes com amostras variáveis | Mudança pareada por semestre como análise principal | Regenerar deterministicamente |
| Semestres | Resumo combinado no paper | Estratificar por semestre; combinado como secundário | Atualizar resultados |
| Identidade | Não usada no CSV publicado | Usar vínculo pseudonimizado somente na análise privada | Implementar proteção de privacidade |
| LLM | Nenhuma chamada | Continuar sem LLM para M2 estruturado | Não gerar custo LLM |
| M2 legado | Resultado principal | Baseline de comparação | Preservar sem sobrescrever |

**Recomendação final:** manter o M2 porque ele adiciona comparação específica entre papéis que o novo M1c não oferece. Refatorar agora a análise dos dados existentes; redesenhar o instrumento na próxima coleta.